# HSK-5 Tutor — train → merge → GGUF (Colab)

End-to-end on a Colab GPU: QLoRA fine-tune of **Qwen2.5-7B-Instruct**, optional before/after eval, merge the adapter, and quantize to a **Q4_K_M GGUF** you download and serve locally with `app.py`.

**Set a GPU first:** *Runtime → Change runtime type → GPU* (L4 or A100; free T4 works but is slow for a 7B).

You'll upload from the repo: `config.py`, `train.py`, `eval.py`, `merge.py`, and the generated `data/train.jsonl` + `data/eval.jsonl`.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install training deps

In [ ]:
!pip install -q -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" \
    "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0" "anthropic>=0.40"

## 3. Upload the project files
Pick `config.py`, `train.py`, `eval.py`, `merge.py`, `train.jsonl`, `eval.jsonl`.

In [ ]:
import os, shutil
from google.colab import files

uploaded = files.upload()
os.makedirs('data', exist_ok=True)
for name in ('train.jsonl', 'eval.jsonl'):
    if name in uploaded:
        shutil.move(name, f'data/{name}')
assert os.path.exists('config.py') and os.path.exists('train.py'), 'need config.py + train.py'
assert os.path.exists('data/train.jsonl'), 'need data/train.jsonl'
print('ready:', sorted(os.listdir('.')), '| data:', sorted(os.listdir('data')))

## 4. Sanity run (optional)
~20 steps to confirm it loads and loss moves.

In [ ]:
!python train.py --max-steps 20

## 5. Full training
~30–60 min on L4/A100. Saves the adapter to `outputs/`.

In [ ]:
!python train.py

## 6. Before/after eval (optional)
Base vs. tuned on held-out prompts → `outputs/eval_report.md`. Add `--judge` to score with Claude (set the key first).

In [ ]:
# import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # only needed for --judge
!python eval.py --n 12

## 7. Merge adapter → fp16 model

In [ ]:
!python merge.py

## 8. Convert + quantize to GGUF (Q4_K_M)
Uses llama.cpp: convert the merged model to f16 GGUF, build the quantizer, then quantize to ~4.7GB Q4_K_M.

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q gguf sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py outputs/merged \
    --outfile outputs/hsk5-tutor-f16.gguf --outtype f16
!cmake -B llama.cpp/build -S llama.cpp -DLLAMA_CURL=OFF > /dev/null 2>&1
!cmake --build llama.cpp/build --config Release -t llama-quantize -j > /dev/null 2>&1
!./llama.cpp/build/bin/llama-quantize \
    outputs/hsk5-tutor-f16.gguf outputs/hsk5-tutor-q4_k_m.gguf Q4_K_M

## 9. Download the GGUF
Put it in your local repo's `outputs/`, then run `python app.py` on the Mac.

In [ ]:
files.download('outputs/hsk5-tutor-q4_k_m.gguf')